In [ ]:
import sys
sys.path.append('..')

import torch
from src import models, data, lens, functional
from src.utils import experiment_utils
from baukit import Menu, show

In [ ]:
import gc
import torch

# Delete the model to free GPU memory
if 'mt' in globals():
    del mt
    print("Model deleted")

# Clear CUDA cache
torch.cuda.empty_cache()

# Force garbage collection
gc.collect()

# Check memory status
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"Max allocated: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")
else:
    print("CUDA not available")

Allocated: 0.00 GB
Reserved: 0.00 GB
Max allocated: 0.00 GB


In [ ]:
device = "cuda:0"
mt = models.load_model("mistral", device=device, fp16=True)
print(f"dtype: {mt.model.dtype}, device: {mt.model.device}, memory: {mt.model.get_memory_footprint()}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

dtype: torch.float16, device: cuda:0, memory: 14496047232


In [ ]:
dataset = data.load_dataset()

relation_names = [r.name for r in dataset.relations]

# Manual selection instead of baukit.Menu (not supported in VS Code)
# Choose a relation by uncommenting one of the lines below or setting relation_name directly
relation_name = relation_names[42]  # Default to first relation
# relation_name = "person_occupation"  # Or specify a relation name directly

print(f"Available relations: {len(relation_names)}")
print(f"Selected: {relation_name}")

# relation_name is set in the previous cell
relation = dataset.filter(relation_names=[relation_name])[0]
print(f"{relation.name} -- {len(relation.samples)} samples")
print("------------------------------------------------------")

experiment_utils.set_seed(12345) # set seed to a constant value for sampling consistency
train, test = relation.split(5)
print("\n".join([sample.__str__() for sample in train.samples]))

Available relations: 47
Selected: adjective antonym
adjective antonym -- 100 samples
------------------------------------------------------
open -> lock
inside -> outside
remember -> forget
close -> open
clockwise -> counterclockwise


In [ ]:
# Print relation information with indices and prompt templates
print("Relations ranked by sample size (with original index):\n")
print(f"{'Index':<8} {'Relation Name':<30} {'Samples':<10} {'Prompt Template'}")
print("-" * 100)

# Create list of (index, relation_name, sample_count, prompt_template)
relation_data = []
for i, name in enumerate(relation_names):
    rel = dataset.filter(relation_names=[name])[0]
    relation_data.append((i, name, len(rel.samples), rel.prompt_templates[0]))

# Sort by sample size (descending)
relation_data_sorted = sorted(relation_data, key=lambda x: x[2], reverse=True)

# Print sorted results
for idx, name, count, template in relation_data_sorted:
    print(f"{idx:<8} {name:<30} {count:<10} {template}")

print(f"\nSelected relation: index {relation_names.index(relation_name)} - {relation_name}")
print(f"Prompt template: {relation.prompt_templates[0]}")


Relations ranked by sample size (with original index):

Index    Relation Name                  Samples    Prompt Template
----------------------------------------------------------------------------------------------------
27       person mother                  994        {}'s mother is named
26       person father                  991        {}'s father is named
31       person sport position          952        {} plays in the position of a
24       landmark on continent          947        {} is on the continent of
28       person native language         919        {} speaks the language of
23       landmark in country            836        {} is in the country of
29       person occupation              821        {} works as a
17       company hq                     674        {} is headquartered in the city of
37       product by company             522        {} was created by
30       person plays instrument        513        {} plays the
38       star constellation name      

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# import gc
# import torch

# # Grid search parameters
# layers = list(range(8, 11))  # 8 to 12 inclusive
# betas = np.arange(1.0, 2.1, 0.1)  # 0.8 to 1.2 in steps of 0.1

# # Store results
# results = []

# print(f"Running grid search over {len(layers)} layers × {len(betas)} betas = {len(layers) * len(betas)} configurations\n")

# # Enable gradient checkpointing to save memory
# mt.model.gradient_checkpointing_enable()

# # Reload the modules to get the latest changes
# import importlib
# import src.operators
# import src.functional
# importlib.reload(src.functional)
# importlib.reload(src.operators)

# from src.operators import JacobianIclMeanEstimator

# # Use a custom string to prepend
# prepend_text = ""

# # Use 5 samples for Jacobian
# jacobian_samples = train.samples[:5]

# for layer in layers:
#     for beta in betas:
#         # Aggressive memory cleanup before each iteration
#         torch.cuda.empty_cache()
#         gc.collect()
        
#         print(f"Testing layer={layer}, beta={beta:.1f}...", end=" ", flush=True)
        
#         estimator = None
#         operator = None
        
#         try:
#             estimator = JacobianIclMeanEstimator(
#                 mt=mt, 
#                 h_layer=layer,
#                 beta=beta,
#                 prepend_string=prepend_text,
#             )
#             operator = estimator(
#                 relation.set(
#                     samples=jacobian_samples,
#                 )
#             )
            
#             # Evaluate on test set
#             correct = 0
#             wrong = 0
#             for sample in test.samples:
#                 predictions = operator(subject=sample.subject).predictions
#                 known_flag = functional.is_nontrivial_prefix(
#                     prediction=predictions[0].token, target=sample.object
#                 )
#                 correct += known_flag
#                 wrong += not known_flag
            
#             faithfulness = correct / (correct + wrong)
#             results.append({
#                 'layer': layer,
#                 'beta': beta,
#                 'faithfulness': faithfulness,
#                 'correct': correct,
#                 'wrong': wrong
#             })
            
#             print(f"Faithfulness = {faithfulness:.3f} ({correct}/{correct+wrong})")
            
#         except Exception as e:
#             print(f"ERROR: {e}")
#             results.append({
#                 'layer': layer,
#                 'beta': beta,
#                 'faithfulness': 0.0,
#                 'correct': 0,
#                 'wrong': len(test.samples)
#             })
        
#         finally:
#             # Clean up operator and estimator explicitly
#             del operator
#             del estimator
#             torch.cuda.empty_cache()
#             gc.collect()

# print("\n" + "="*60)
# print("Grid Search Complete!")
# print("="*60)

# # Create visualization
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# # Prepare data for heatmap
# faithfulness_matrix = np.zeros((len(betas), len(layers)))
# for result in results:
#     beta_idx = np.argmin(np.abs(betas - result['beta']))
#     layer_idx = layers.index(result['layer'])
#     faithfulness_matrix[beta_idx, layer_idx] = result['faithfulness']

# # Heatmap
# im = ax1.imshow(faithfulness_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
# ax1.set_xticks(range(len(layers)))
# ax1.set_xticklabels(layers)
# ax1.set_yticks(range(len(betas)))
# ax1.set_yticklabels([f'{b:.1f}' for b in betas])
# ax1.set_xlabel('Layer')
# ax1.set_ylabel('Beta')
# ax1.set_title('Faithfulness Heatmap')
# plt.colorbar(im, ax=ax1, label='Faithfulness')

# # Add text annotations
# for i in range(len(betas)):
#     for j in range(len(layers)):
#         text = ax1.text(j, i, f'{faithfulness_matrix[i, j]:.2f}',
#                        ha="center", va="center", color="black", fontsize=8)

# # Line plot for each layer
# for layer in layers:
#     layer_results = [r for r in results if r['layer'] == layer]
#     layer_betas = [r['beta'] for r in layer_results]
#     layer_faithfulness = [r['faithfulness'] for r in layer_results]
#     ax2.plot(layer_betas, layer_faithfulness, marker='o', label=f'Layer {layer}')

# ax2.set_xlabel('Beta')
# ax2.set_ylabel('Faithfulness')
# ax2.set_title('Faithfulness vs Beta (by Layer)')
# ax2.legend()
# ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# # Print best configuration
# best_result = max(results, key=lambda x: x['faithfulness'])
# print(f"\nBest Configuration:")
# print(f"  Layer: {best_result['layer']}")
# print(f"  Beta: {best_result['beta']:.1f}")
# print(f"  Faithfulness: {best_result['faithfulness']:.3f} ({best_result['correct']}/{best_result['correct']+best_result['wrong']})")

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# import gc
# import torch

# # Grid search parameters
# layers = list(range(8, 13))  # 8 to 12 inclusive
# betas = np.arange(1.6, 2.5, 0.1)  # 0.8 to 1.2 in steps of 0.1

# # Store results
# results = []

# print(f"Running grid search over {len(layers)} layers × {len(betas)} betas = {len(layers) * len(betas)} configurations\n")

# # Enable gradient checkpointing to save memory
# mt.model.gradient_checkpointing_enable()

# # Reload the modules to get the latest changes
# import importlib
# import src.operators
# import src.functional
# importlib.reload(src.functional)
# importlib.reload(src.operators)

# from src.operators import JacobianIclMeanEstimator

# # Use a custom string to prepend
# prepend_text = ""

# # Use 5 samples for Jacobian
# jacobian_samples = train.samples[:5]

# for layer in layers:
#     for beta in betas:
#         # Aggressive memory cleanup before each iteration
#         torch.cuda.empty_cache()
#         gc.collect()
        
#         print(f"Testing layer={layer}, beta={beta:.1f}...", end=" ", flush=True)
        
#         estimator = None
#         operator = None
        
#         try:
#             estimator = JacobianIclMeanEstimator(
#                 mt=mt, 
#                 h_layer=layer,
#                 beta=beta,
#                 prepend_string=prepend_text,
#             )
#             operator = estimator(
#                 relation.set(
#                     samples=jacobian_samples,
#                 )
#             )
            
#             # Evaluate on test set
#             correct = 0
#             wrong = 0
#             for sample in test.samples:
#                 predictions = operator(subject=sample.subject).predictions
#                 known_flag = functional.is_nontrivial_prefix(
#                     prediction=predictions[0].token, target=sample.object
#                 )
#                 correct += known_flag
#                 wrong += not known_flag
            
#             faithfulness = correct / (correct + wrong)
#             results.append({
#                 'layer': layer,
#                 'beta': beta,
#                 'faithfulness': faithfulness,
#                 'correct': correct,
#                 'wrong': wrong
#             })
            
#             print(f"Faithfulness = {faithfulness:.3f} ({correct}/{correct+wrong})")
            
#         except Exception as e:
#             print(f"ERROR: {e}")
#             results.append({
#                 'layer': layer,
#                 'beta': beta,
#                 'faithfulness': 0.0,
#                 'correct': 0,
#                 'wrong': len(test.samples)
#             })
        
#         finally:
#             # Clean up operator and estimator explicitly
#             del operator
#             del estimator
#             torch.cuda.empty_cache()
#             gc.collect()

# print("\n" + "="*60)
# print("Grid Search Complete!")
# print("="*60)

# # Create visualization
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# # Prepare data for heatmap
# faithfulness_matrix = np.zeros((len(betas), len(layers)))
# for result in results:
#     beta_idx = np.argmin(np.abs(betas - result['beta']))
#     layer_idx = layers.index(result['layer'])
#     faithfulness_matrix[beta_idx, layer_idx] = result['faithfulness']

# # Heatmap
# im = ax1.imshow(faithfulness_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
# ax1.set_xticks(range(len(layers)))
# ax1.set_xticklabels(layers)
# ax1.set_yticks(range(len(betas)))
# ax1.set_yticklabels([f'{b:.1f}' for b in betas])
# ax1.set_xlabel('Layer')
# ax1.set_ylabel('Beta')
# ax1.set_title('Faithfulness Heatmap')
# plt.colorbar(im, ax=ax1, label='Faithfulness')

# # Add text annotations
# for i in range(len(betas)):
#     for j in range(len(layers)):
#         text = ax1.text(j, i, f'{faithfulness_matrix[i, j]:.2f}',
#                        ha="center", va="center", color="black", fontsize=8)

# # Line plot for each layer
# for layer in layers:
#     layer_results = [r for r in results if r['layer'] == layer]
#     layer_betas = [r['beta'] for r in layer_results]
#     layer_faithfulness = [r['faithfulness'] for r in layer_results]
#     ax2.plot(layer_betas, layer_faithfulness, marker='o', label=f'Layer {layer}')

# ax2.set_xlabel('Beta')
# ax2.set_ylabel('Faithfulness')
# ax2.set_title('Faithfulness vs Beta (by Layer)')
# ax2.legend()
# ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# # Print best configuration
# best_result = max(results, key=lambda x: x['faithfulness'])
# print(f"\nBest Configuration:")
# print(f"  Layer: {best_result['layer']}")
# print(f"  Beta: {best_result['beta']:.1f}")
# print(f"  Faithfulness: {best_result['faithfulness']:.3f} ({best_result['correct']}/{best_result['correct']+best_result['wrong']})")

In [ ]:
################### hparams ###################
layer = 8
beta = 2.4
###############################################

In [ ]:
# Enable gradient checkpointing to save memory can remove later not sure if relevant
mt.model.gradient_checkpointing_enable()

In [ ]:
# Clear memory before creating operator
import gc
torch.cuda.empty_cache()
gc.collect()

# Reload the modules to get the latest changes
import importlib
import src.operators
import src.functional
importlib.reload(src.functional)
importlib.reload(src.operators)

from src.operators import JacobianIclMeanEstimator

# Use a custom string to prepend
prepend_text = "It is opposite day. Take the opposite of the following \n"

# Try 5 samples again now that we clear memory after each Jacobian
jacobian_samples = train.samples[:5]

print(f"Prepended text (not trained on): {prepend_text.strip()}")
print(f"\nJacobian computed over {len(jacobian_samples)} samples:")
for i, s in enumerate(jacobian_samples, 1):
    print(f"  {i}. {s}")

estimator = JacobianIclMeanEstimator(
    mt = mt, 
    h_layer = layer,
    beta = beta,
    prepend_string=prepend_text,
)
operator = estimator(
    relation.set(
        samples=jacobian_samples,  # Jacobian computed over these (leave-one-out)
    )
)

print(f"\n✓ Operator created successfully")
print(f"  Structure: Custom prepended text + {len(jacobian_samples)} for Jacobian (leave-one-out)")
print(f"\nPrompt structure during training:")
print(f"  {prepend_text.strip()}  [Prepended: NOT in Jacobian]")
for i, s in enumerate(jacobian_samples, 1):
    print(f"  {s.subject} -> {s.object}  [Sample {i}: leave-one-out]")
print(f"  [Query subject]")

Prepended text (not trained on): It is opposite day. Take the opposite of the following

Jacobian computed over 5 samples:
  1. open -> lock
  2. inside -> outside
  3. remember -> forget
  4. close -> open
  5. clockwise -> counterclockwise

✓ Operator created successfully
  Structure: Custom prepended text + 5 for Jacobian (leave-one-out)

Prompt structure during training:
  It is opposite day. Take the opposite of the following  [Prepended: NOT in Jacobian]
  open -> lock  [Sample 1: leave-one-out]
  inside -> outside  [Sample 2: leave-one-out]
  remember -> forget  [Sample 3: leave-one-out]
  close -> open  [Sample 4: leave-one-out]
  clockwise -> counterclockwise  [Sample 5: leave-one-out]
  [Query subject]


# Checking $faithfulness$

In [ ]:
test = functional.filter_relation_samples_based_on_provided_fewshots(
    mt=mt, test_relation=test, prompt_template=operator.prompt_template, batch_size=4
)
sample = test.samples[0]
print(sample)

operator(subject = sample.subject).predictions

hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject],
    h_layer= operator.h_layer,
)

h = hs_and_zs.h_by_subj[sample.subject]

z = operator.beta * (operator.weight @ h) + operator.bias

lens.logit_lens(
    mt = mt,
    h = z,
    get_proba = True
)

correct = 0
wrong = 0
for sample in test.samples:
    predictions = operator(subject = sample.subject).predictions
    known_flag = functional.is_nontrivial_prefix(
        prediction=predictions[0].token, target=sample.object
    )
    print(f"{sample.subject=}, {sample.object=}, ", end="")
    print(f'predicted="{functional.format_whitespace(predictions[0].token)}", (p={predictions[0].prob}), known=({functional.get_tick_marker(known_flag)})')
    
    correct += known_flag
    wrong += not known_flag
    
faithfulness = correct/(correct + wrong)

print("------------------------------------------------------------")
print(f"Faithfulness (@1) = {faithfulness}")
print("------------------------------------------------------------")

accept -> reject
sample.subject='accept', sample.object='reject', predicted="not", (p=0.07252021133899689), known=(✗)
sample.subject='arrive', sample.object='depart', predicted="not", (p=0.07075457274913788), known=(✗)
sample.subject='attract', sample.object='repel', predicted="not", (p=0.07835444808006287), known=(✗)
sample.subject='beautiful', sample.object='ugly', predicted="not", (p=0.1168309897184372), known=(✗)
sample.subject='bend', sample.object='straighten', predicted="not", (p=0.06392142176628113), known=(✗)
sample.subject='big', sample.object='small', predicted="not", (p=0.14813247323036194), known=(✗)
sample.subject='bless', sample.object='curse', predicted="not", (p=0.09525701403617859), known=(✗)
sample.subject='borrow', sample.object='lend', predicted="not", (p=0.07468985766172409), known=(✗)
sample.subject='brave', sample.object='cowardly', predicted="not", (p=0.13039030134677887), known=(✗)
sample.subject='buy', sample.object='sell', predicted="not", (p=0.0624109208583

In [ ]:
perturb = operator.weight
perturb 


tensor([[-7.5378e-03,  1.2970e-02, -1.7366e-03,  ..., -1.9882e-02,
          2.7069e-02,  1.2772e-02],
        [ 8.3008e-03,  1.4755e-02,  9.3079e-03,  ...,  2.9572e-02,
          4.4746e-03, -1.4572e-02],
        [ 1.4893e-02, -1.6678e-02, -3.2692e-03,  ..., -1.0406e-02,
         -1.9043e-02,  1.2680e-02],
        ...,
        [ 2.5826e-03, -7.9334e-05,  5.6000e-03,  ...,  1.8967e-02,
         -1.7899e-02,  2.0199e-03],
        [-1.0162e-02,  3.5038e-03, -8.4381e-03,  ..., -7.3357e-03,
          2.6428e-02,  9.0103e-03],
        [-8.6365e-03, -2.8503e-02,  4.8981e-02,  ...,  9.9030e-03,
         -4.2648e-03,  9.1171e-03]], device='cuda:0', dtype=torch.float16)

In [ ]:
# Clear memory before creating operator
torch.cuda.empty_cache()
gc.collect()

print("Memory cleared")
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

Memory cleared
Allocated: 15.31 GB
Reserved: 15.36 GB


In [ ]:
# Clear memory before creating operator
import gc
torch.cuda.empty_cache()
gc.collect()

# Reload the modules to get the latest changes
import importlib
import src.operators
import src.functional
importlib.reload(src.functional)
importlib.reload(src.operators)

from src.operators import JacobianIclMeanEstimator

# Use a custom string to prepend
prepend_text = ""

# Try 5 samples again now that we clear memory after each Jacobian
jacobian_samples = train.samples[:5]

print(f"Prepended text (not trained on): {prepend_text.strip()}")
print(f"\nJacobian computed over {len(jacobian_samples)} samples:")
for i, s in enumerate(jacobian_samples, 1):
    print(f"  {i}. {s}")

estimator = JacobianIclMeanEstimator(
    mt = mt, 
    h_layer = layer,
    beta = beta,
    prepend_string=prepend_text,
)
operator = estimator(
    relation.set(
        samples=jacobian_samples,  # Jacobian computed over these (leave-one-out)
    )
)

print(f"\n✓ Operator created successfully")
print(f"  Structure: Custom prepended text + {len(jacobian_samples)} for Jacobian (leave-one-out)")
print(f"\nPrompt structure during training:")
print(f"  {prepend_text.strip()}  [Prepended: NOT in Jacobian]")
for i, s in enumerate(jacobian_samples, 1):
    print(f"  {s.subject} -> {s.object}  [Sample {i}: leave-one-out]")
print(f"  [Query subject]")

Prepended text (not trained on): 

Jacobian computed over 5 samples:
  1. open -> lock
  2. inside -> outside
  3. remember -> forget
  4. close -> open
  5. clockwise -> counterclockwise

✓ Operator created successfully
  Structure: Custom prepended text + 5 for Jacobian (leave-one-out)

Prompt structure during training:
    [Prepended: NOT in Jacobian]
  open -> lock  [Sample 1: leave-one-out]
  inside -> outside  [Sample 2: leave-one-out]
  remember -> forget  [Sample 3: leave-one-out]
  close -> open  [Sample 4: leave-one-out]
  clockwise -> counterclockwise  [Sample 5: leave-one-out]
  [Query subject]


In [ ]:

normal = operator.weight

normal 

tensor([[ 0.0067, -0.0021, -0.0054,  ..., -0.0074,  0.0134, -0.0077],
        [ 0.0073,  0.0130,  0.0084,  ...,  0.0169,  0.0018, -0.0086],
        [ 0.0143, -0.0142, -0.0069,  ..., -0.0100, -0.0276, -0.0012],
        ...,
        [ 0.0237,  0.0052,  0.0028,  ...,  0.0217, -0.0189,  0.0087],
        [-0.0079, -0.0013, -0.0028,  ...,  0.0009,  0.0253, -0.0049],
        [-0.0090, -0.0184,  0.0211,  ...,  0.0025, -0.0060,  0.0093]],
       device='cuda:0', dtype=torch.float16)

In [ ]:
test = functional.filter_relation_samples_based_on_provided_fewshots(
    mt=mt, test_relation=test, prompt_template=operator.prompt_template, batch_size=4
)
sample = test.samples[0]
print(sample)

operator(subject = sample.subject).predictions

hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject],
    h_layer= operator.h_layer,
)

h = hs_and_zs.h_by_subj[sample.subject]

z = operator.beta * (operator.weight @ h) + operator.bias

lens.logit_lens(
    mt = mt,
    h = z,
    get_proba = True
)

correct = 0
wrong = 0
for sample in test.samples:
    predictions = operator(subject = sample.subject).predictions
    known_flag = functional.is_nontrivial_prefix(
        prediction=predictions[0].token, target=sample.object
    )
    print(f"{sample.subject=}, {sample.object=}, ", end="")
    print(f'predicted="{functional.format_whitespace(predictions[0].token)}", (p={predictions[0].prob}), known=({functional.get_tick_marker(known_flag)})')
    
    correct += known_flag
    wrong += not known_flag
    
faithfulness = correct/(correct + wrong)

print("------------------------------------------------------------")
print(f"Faithfulness (@1) = {faithfulness}")
print("------------------------------------------------------------")

accept -> reject
sample.subject='accept', sample.object='reject', predicted="not", (p=0.1472734957933426), known=(✗)
sample.subject='arrive', sample.object='depart', predicted="\n", (p=0.2403177171945572), known=(✗)
sample.subject='attract', sample.object='repel', predicted="\n", (p=0.1275210976600647), known=(✗)
sample.subject='beautiful', sample.object='ugly', predicted="not", (p=0.11982479691505432), known=(✗)
sample.subject='bend', sample.object='straighten', predicted="\n", (p=0.12243638187646866), known=(✗)
sample.subject='big', sample.object='small', predicted="not", (p=0.23850072920322418), known=(✗)
sample.subject='bless', sample.object='curse', predicted="to", (p=0.1428741216659546), known=(✗)
sample.subject='borrow', sample.object='lend', predicted="\n", (p=0.17323963344097137), known=(✗)
sample.subject='brave', sample.object='cowardly', predicted="not", (p=0.17526978254318237), known=(✗)
sample.subject='buy', sample.object='sell', predicted="not", (p=0.0938497856259346), kn

In [ ]:
# Compute Frobenius norm of the difference between normal and perturb
frobenius_norm = torch.norm(normal - perturb, p='fro')

print(f"Frobenius norm between normal and perturb: {frobenius_norm:.4f}")
print(f"Normal weight shape: {normal.shape}")
print(f"Perturb weight shape: {perturb.shape}")
print(f"Relative difference: {(frobenius_norm / torch.norm(normal, p='fro')):.4f}")

Frobenius norm between normal and perturb: 54.3750
Normal weight shape: torch.Size([4096, 4096])
Perturb weight shape: torch.Size([4096, 4096])
Relative difference: 0.9585


In [ ]:
# Compute eigenvalues of both matrices
eigenvalues_normal = torch.linalg.eigvals(normal.float())
eigenvalues_perturb = torch.linalg.eigvals(perturb.float())

print("Normal matrix eigenvalues:")
print(f"  Shape: {eigenvalues_normal.shape}")
print(f"  Real part - Min: {eigenvalues_normal.real.min():.4f}, Max: {eigenvalues_normal.real.max():.4f}")
print(f"  Real part - Mean: {eigenvalues_normal.real.mean():.4f}, Std: {eigenvalues_normal.real.std():.4f}")
print(f"  Top 5 by magnitude: {torch.abs(eigenvalues_normal).topk(5).values}")

print("\nPerturb matrix eigenvalues:")
print(f"  Shape: {eigenvalues_perturb.shape}")
print(f"  Real part - Min: {eigenvalues_perturb.real.min():.4f}, Max: {eigenvalues_perturb.real.max():.4f}")
print(f"  Real part - Mean: {eigenvalues_perturb.real.mean():.4f}, Std: {eigenvalues_perturb.real.std():.4f}")
print(f"  Top 5 by magnitude: {torch.abs(eigenvalues_perturb).topk(5).values}")

# Compare eigenvalue distributions
print("\nComparison:")
print(f"  Difference in top eigenvalue magnitude: {(torch.abs(eigenvalues_normal).max() - torch.abs(eigenvalues_perturb).max()):.4f}")

Normal matrix eigenvalues:
  Shape: torch.Size([4096])
  Real part - Min: -0.7633, Max: 1.1090
  Real part - Mean: 0.0144, Std: 0.1460
  Top 5 by magnitude: tensor([1.1090, 0.9192, 0.9192, 0.8519, 0.8519], device='cuda:0')

Perturb matrix eigenvalues:
  Shape: torch.Size([4096])
  Real part - Min: -1.1748, Max: 0.9868
  Real part - Mean: 0.0152, Std: 0.1652
  Top 5 by magnitude: tensor([1.1990, 1.1990, 1.1864, 1.1864, 1.0554], device='cuda:0')

Comparison:
  Difference in top eigenvalue magnitude: -0.0901


In [ ]:
z = operator.beta * (operator.weight @ h) + operator.bias

lens.logit_lens(
    mt = mt,
    h = z,
    get_proba = True
)

([('not', 0.147),
  ('\n', 0.127),
  ('to', 0.052),
  ('...', 0.04),
  (':', 0.036),
  ('opposite', 0.025),
  ('open', 0.022),
  ('the', 0.021),
  ('a', 0.02),
  ('(', 0.017)],
 {})

In [ ]:
# Store the operator weight and bias for later analysis
operator_weight = operator.weight.clone().detach()
operator_bias = operator.bias.clone().detach()

print(f"Operator weight shape: {operator_weight.shape}")
print(f"Operator bias shape: {operator_bias.shape}")
print(f"Weight dtype: {operator_weight.dtype}, device: {operator_weight.device}")
print(f"Bias dtype: {operator_bias.dtype}, device: {operator_bias.device}")

Operator weight shape: torch.Size([4096, 4096])
Operator bias shape: torch.Size([1, 4096])
Weight dtype: torch.float16, device: cuda:0
Bias dtype: torch.float16, device: cuda:0


In [ ]:
# Clear CUDA memory from previous calculations
torch.cuda.empty_cache()
gc.collect()

print("Memory cleared")
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

Memory cleared
Allocated: 15.00 GB
Reserved: 16.08 GB


In [ ]:
# Enable gradient checkpointing to save memory can remove later not sure if relevant
mt.model.gradient_checkpointing_enable()

# Clear memory before creating operator
import gc
torch.cuda.empty_cache()
gc.collect()

# Reload the modules to get the latest changes
import importlib
import src.operators
import src.functional
importlib.reload(src.functional)
importlib.reload(src.operators)

from src.operators import JacobianIclMeanEstimator

# Use a custom string to prepend
prepend_text = ""

# Try 5 samples again now that we clear memory after each Jacobian
jacobian_samples = train.samples[:5]

print(f"Prepended text (not trained on): {prepend_text.strip()}")
print(f"\nJacobian computed over {len(jacobian_samples)} samples:")
for i, s in enumerate(jacobian_samples, 1):
    print(f"  {i}. {s}")

estimator = JacobianIclMeanEstimator(
    mt = mt, 
    h_layer = layer,
    beta = beta,
    prepend_string=prepend_text,
)
operator_new = estimator(
    relation.set(
        samples=jacobian_samples,  # Jacobian computed over these (leave-one-out)
    )
)

print(f"\n✓ Operator created successfully")
print(f"  Structure: Custom prepended text + {len(jacobian_samples)} for Jacobian (leave-one-out)")
print(f"\nPrompt structure during training:")
print(f"  {prepend_text.strip()}  [Prepended: NOT in Jacobian]")
for i, s in enumerate(jacobian_samples, 1):
    print(f"  {s.subject} -> {s.object}  [Sample {i}: leave-one-out]")
print(f"  [Query subject]")

test_new = functional.filter_relation_samples_based_on_provided_fewshots(
    mt=mt, test_relation=test, prompt_template=operator_new.prompt_template, batch_size=4
)

sample_new = test_new.samples[0]
print(sample_new)
operator_new(subject = sample_new.subject).predictions

z_new = operator_new.beta * (operator_new.weight @ h) + operator_new.bias

lens.logit_lens(
    mt = mt,
    h = z_new,
    get_proba = True
)

correct = 0
wrong = 0
for sample_iter in test_new.samples:
    predictions = operator_new(subject = sample_iter.subject).predictions
    known_flag = functional.is_nontrivial_prefix(
        prediction=predictions[0].token, target=sample_iter.object
    )
    print(f"{sample_iter.subject=}, {sample_iter.object=}, ", end="")
    print(f'predicted="{functional.format_whitespace(predictions[0].token)}", (p={predictions[0].prob}), known=({functional.get_tick_marker(known_flag)})')
    
    correct += known_flag
    wrong += not known_flag
    
faithfulness_new = correct/(correct + wrong)

print("------------------------------------------------------------")
print(f"Faithfulness (@1) = {faithfulness_new}")
print("------------------------------------------------------------")

Prepended text (not trained on): 

Jacobian computed over 5 samples:
  1. open -> lock
  2. inside -> outside
  3. remember -> forget
  4. close -> open
  5. clockwise -> counterclockwise


KeyboardInterrupt: 

In [ ]:
torch.cuda.empty_cache()

# Check memory status
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print("Memory cleared, ready to create new operator")

Allocated: 14.95 GB
Reserved: 16.48 GB
Memory cleared, ready to create new operator


# $causality$

In [ ]:
################### hparams ###################
rank = 100
###############################################

In [ ]:
experiment_utils.set_seed(12345) # set seed to a constant value for sampling consistency
test_targets = functional.random_edit_targets(test.samples)

## setup

In [ ]:
source = test.samples[0]
target = test_targets[source]

f"Changing the mapping ({source}) to ({source.subject} -> {target.object})"

'Changing the mapping (Argentina -> Buenos Aires) to (Argentina -> Riyadh)'

### Calculate $\Delta \mathbf{s}$ such that $\mathbf{s} + \Delta \mathbf{s} \approx \mathbf{s}'$

<p align="center">
    <img align="center" src="causality-crop.png" style="width:80%;"/>
</p>

Under the relation $r =\, $*plays the instrument*, and given the subject $s =\, $*Miles Davis*, the model will predict $o =\, $*trumpet* **(a)**; and given the subject $s' =\, $*Cat Stevens*, the model will now predict $o' =\, $*guiter* **(b)**. 

If the computation from $\mathbf{s}$ to $\mathbf{o}$ is well-approximated by $operator$ parameterized by $W_r$ and $b_r$ **(c)**, then $\Delta{\mathbf{s}}$ **(d)** should tell us the direction of change from $\mathbf{s}$ to $\mathbf{s}'$. Thus, $\tilde{\mathbf{s}}=\mathbf{s}+\Delta\mathbf{s}$ would be an approximation of $\mathbf{s}'$ and patching $\tilde{\mathbf{s}}$ in place of $\mathbf{s}$ should change the prediction to $o'$ = *guitar* 

In [ ]:
def get_delta_s(
    operator, 
    source_subject, 
    target_subject,
    rank = 100,
    fix_latent_norm = None, # if set, will fix the norms of z_source and z_target
):
    w_p_inv = functional.low_rank_pinv(
        matrix = operator.weight,
        rank=rank,
    )
    hs_and_zs = functional.compute_hs_and_zs(
        mt = mt,
        prompt_template = operator.prompt_template,
        subjects = [source_subject, target_subject],
        h_layer= operator.h_layer,
        z_layer=-1,
    )

    z_source = hs_and_zs.z_by_subj[source_subject]
    z_target = hs_and_zs.z_by_subj[target_subject]
    
    z_source *= fix_latent_norm / z_source.norm() if fix_latent_norm is not None else 1.0
    z_target *= z_source.norm() / z_target.norm() if fix_latent_norm is not None else 1.0

    delta_s = w_p_inv @  (z_target.squeeze() - z_source.squeeze())

    return delta_s, hs_and_zs

delta_s, hs_and_zs = get_delta_s(
    operator = operator,
    source_subject = source.subject,
    target_subject = target.subject,
    rank = rank
)

In [ ]:
import baukit

def get_intervention(h, int_layer, subj_idx):
    def edit_output(output, layer):
        if(layer != int_layer):
            return output
        functional.untuple(output)[:, subj_idx] = h 
        return output
    return edit_output

prompt = operator.prompt_template.format(source.subject)

h_index, inputs = functional.find_subject_token_index(
    mt=mt,
    prompt=prompt,
    subject=source.subject,
)

h_layer, z_layer = models.determine_layer_paths(model = mt, layers = [layer, -1])

with baukit.TraceDict(
    mt.model, layers = [h_layer, z_layer],
    edit_output=get_intervention(
#         h = hs_and_zs.h_by_subj[source.subject],         # let the computation proceed as usual
        h = hs_and_zs.h_by_subj[source.subject] + delta_s, # replace s with s + delta_s
        int_layer = h_layer, 
        subj_idx = h_index
    )
) as traces:
    outputs = mt.model(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
    )

lens.interpret_logits(
    mt = mt, 
    logits = outputs.logits[0][-1], 
    get_proba=True
)

[(' Riyadh', 0.802),
 (' J', 0.051),
 (' Mecca', 0.041),
 (' Saudi', 0.012),
 (' Riy', 0.01),
 ('\n', 0.007),
 (' Dam', 0.005),
 (' Cairo', 0.004),
 (' the', 0.004),
 (' Al', 0.003)]

## Measuring causality

In [ ]:
from src.editors import LowRankPInvEditor

svd = torch.svd(operator.weight.float())
editor = LowRankPInvEditor(
    lre=operator,
    rank=rank,
    svd=svd,
)

In [ ]:
# precomputing latents to speed things up
hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject for sample in test.samples],
    h_layer= operator.h_layer,
    z_layer=-1,
    batch_size = 2
)

success = 0
fails = 0

for sample in test.samples:
    target = test_targets.get(sample)
    assert target is not None
    edit_result = editor(
        subject = sample.subject,
        target = target.subject
    )
    
    success_flag = functional.is_nontrivial_prefix(
        prediction=edit_result.predicted_tokens[0].token, target=target.object
    )
    
    print(f"Mapping {sample.subject} -> {target.object} | edit result={edit_result.predicted_tokens[0]} | success=({functional.get_tick_marker(success_flag)})")
    
    success += success_flag
    fails += not success_flag
    
causality = success / (success + fails)

print("------------------------------------------------------------")
print(f"Causality (@1) = {causality}")
print("------------------------------------------------------------")

Mapping Argentina -> Riyadh | edit result= Riyadh (p=0.819) | success=(✓)
Mapping Australia -> Buenos Aires | edit result= Buenos (p=0.822) | success=(✓)
Mapping Canada -> Abuja | edit result= Abu (p=0.610) | success=(✓)
Mapping Chile -> Lima | edit result= Lima (p=0.967) | success=(✓)
Mapping Colombia -> Berlin | edit result= Berlin (p=0.953) | success=(✓)
Mapping Egypt -> Mexico City | edit result= Mexico (p=0.983) | success=(✓)
Mapping France -> Riyadh | edit result= Riyadh (p=0.847) | success=(✓)
Mapping Germany -> Cairo | edit result= Cairo (p=0.970) | success=(✓)
Mapping India -> Lima | edit result= Lima (p=0.930) | success=(✓)
Mapping Mexico -> Santiago | edit result= Santiago (p=0.955) | success=(✓)
Mapping Nigeria -> Riyadh | edit result= Riyadh (p=0.849) | success=(✓)
Mapping Pakistan -> New Delhi | edit result= New (p=0.863) | success=(✓)
Mapping Peru -> Caracas | edit result= Car (p=0.937) | success=(✓)
Mapping Russia -> Cairo | edit result= Cairo (p=0.966) | success=(✓)
Ma